# The Snitch — Full Proof Training Run (Qwen2.5-1.5B, GRPO + LoRA, 400 steps)

This notebook is the end-to-end proof run for The Snitch submission.

It trains a Qwen2.5-1.5B overseer with GRPO + LoRA, plots reward/loss/KL, evaluates against a base-model A/B control, and uploads the adapter only after evaluation.

Recommended hardware: Google Colab Pro A100 or L4. Expected runtime: roughly 1.5–2.5 hours depending on GPU availability and Hub download speed.

## What changed from the earlier 300-step notebook

- Uses `Qwen/Qwen2.5-1.5B-Instruct` with LoRA, same model family as the README result.
- Patches `LEARNING_RATE` from `5e-6` to `2e-5` before training, because the previous run had very low KL and likely under-updated the policy.
- Runs `400` GRPO steps instead of `300`.
- Evaluates all `120` held-out v3 traces instead of the script default `60` sampled traces.
- Sorts checkpoints numerically, so `checkpoint-400` is selected instead of accidentally selecting `checkpoint-50`.
- Runs LoRA-vs-base evaluation before upload, so only a useful run gets published.

## Outputs

- Training curves: `/content/snitch-env/runs/grpo_lr2e5_400steps/training_curves.png`
- LoRA eval JSON: `/content/snitch-env/results/eval_lora_lr2e5_400.json`
- Base eval JSON: `/content/snitch-env/results/eval_base_full120.json`
- Optional model repo: `https://huggingface.co/Mihir1107/snitch-overseer-lr2e5-ckpt400`


## Cell 1 — Install dependencies and clone the repo

This cell makes reruns idempotent, installs training dependencies, and applies the Colab `torchao` compatibility fix.


In [ ]:
!pip install -q --upgrade transformers trl peft accelerate datasets bitsandbytes huggingface_hub matplotlib
!pip uninstall -y -q torchao
!pip install -q "torchao>=0.16.0"

!rm -rf /content/snitch-env
!git clone https://github.com/Mihir1107/snitch-env.git /content/snitch-env
%cd /content/snitch-env
!pip install -q -e ".[training]"

!git rev-parse HEAD
!nvidia-smi


## Cell 2 — Patch training/eval scripts for the proof run

The repository defaults preserve the original conservative run. For the proof run, we patch only two things:

- `LEARNING_RATE = 2e-5` so GRPO can move the LoRA policy more than the previous low-KL run.
- `gen_gap_eval.py` default sample size from `60` to `120`, so the reported v3 evaluation covers all held-out traces.


In [ ]:
from pathlib import Path

train_script = Path('scripts/train_easy_only.py')
s = train_script.read_text()
if 'LEARNING_RATE = 5e-6' not in s and 'LEARNING_RATE = 2e-5' not in s:
    raise RuntimeError('Could not find expected LEARNING_RATE line in scripts/train_easy_only.py')
s = s.replace('LEARNING_RATE = 5e-6', 'LEARNING_RATE = 2e-5')
train_script.write_text(s)
assert 'LEARNING_RATE = 2e-5' in train_script.read_text(), 'LR patch failed!'
assert 'LEARNING_RATE = 5e-6' not in train_script.read_text(), 'old LR still present!'

eval_script = Path('scripts/gen_gap_eval.py')
s = eval_script.read_text()
old_sig = 'def load_and_sample_traces(path: str, n: int = 60, seed: int = 42)'
new_sig = 'def load_and_sample_traces(path: str, n: int = 120, seed: int = 42)'
if old_sig not in s and new_sig not in s:
    raise RuntimeError('Could not find expected load_and_sample_traces signature in scripts/gen_gap_eval.py')
s = s.replace(old_sig, new_sig)
eval_script.write_text(s)
assert new_sig in eval_script.read_text(), 'Eval sample-size patch failed!'

print('Patched scripts/train_easy_only.py: LEARNING_RATE = 2e-5')
print('Patched scripts/gen_gap_eval.py: eval sample size = 120')

# Quick verification
print('--- train config lines ---')
for line in train_script.read_text().splitlines():
    if line.startswith(('LEARNING_RATE', 'BETA', 'MAX_COMPLETION_LENGTH', 'NUM_GENERATIONS')):
        print(line)
print('--- eval sample signature ---')
for line in eval_script.read_text().splitlines():
    if line.startswith('def load_and_sample_traces'):
        print(line)


## Cell 3 — Inspect the training and held-out evaluation data

Confirms the train/eval split is correct:

- Train: 960 traces, variants 1+2, 120 unique question IDs
- Eval: 120 traces, variant 3 only, 30 unique question IDs
- Train/eval question IDs are disjoint


In [ ]:
import json
from collections import Counter

with open('data/train_traces.jsonl') as f:
    train = [json.loads(l) for l in f]
with open('data/eval_traces.jsonl') as f:
    eval_ = [json.loads(l) for l in f]

train_qids = set(t['question_id'] for t in train)
eval_qids = set(t['question_id'] for t in eval_)

print(f'Train traces: {len(train)}')
print(f'  Classes: {dict(Counter(t["policy_class"] for t in train))}')
print(f'  Variants: {dict(Counter(t["policy_variant"] for t in train))}')
print(f'  Unique question IDs: {len(train_qids)}')
print()
print(f'Eval traces (held-out v3): {len(eval_)}')
print(f'  Classes: {dict(Counter(t["policy_class"] for t in eval_))}')
print(f'  Variants: {dict(Counter(t["policy_variant"] for t in eval_))}')
print(f'  Unique question IDs: {len(eval_qids)}')
print()
print(f'Train ∩ Eval question IDs: {len(train_qids & eval_qids)} (must be 0)')


## Cell 4 — Run GRPO training

This is the long proof run.

Expected runtime on Colab Pro A100: about 1.5–2.5 hours. The run saves checkpoints every 50 steps and a final adapter under `/content/snitch-env/runs/grpo_lr2e5_400steps/final`.


In [ ]:
!python scripts/train_easy_only.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --variants "1,2" \
  --train-path data/train_traces.jsonl \
  --max-steps 400 \
  --logging-steps 10 \
  --save-steps 50 \
  --eval-steps 50 \
  --output-dir /content/snitch-env/runs/grpo_lr2e5_400steps


## Cell 5 — Plot training curves

Reads `trainer_state.json` from the latest numeric checkpoint and plots reward, loss, KL, and eval reward. Numeric checkpoint sorting avoids accidentally selecting `checkpoint-50` over `checkpoint-400`.


In [ ]:
import json
import glob
import re
import matplotlib.pyplot as plt

RUN_DIR = '/content/snitch-env/runs/grpo_lr2e5_400steps'

def ckpt_step(path):
    m = re.search(r'checkpoint-(\d+)$', path)
    return int(m.group(1)) if m else -1

ckpts = sorted(glob.glob(f'{RUN_DIR}/checkpoint-*'), key=ckpt_step)
if not ckpts:
    raise FileNotFoundError('No checkpoints found — training cell may have failed')

latest_ckpt = ckpts[-1]
trainer_state_path = f'{latest_ckpt}/trainer_state.json'
print(f'Reading: {trainer_state_path}')

with open(trainer_state_path) as f:
    log = json.load(f)['log_history']

reward_entries = [(e['step'], e['reward']) for e in log if 'reward' in e]
loss_entries = [(e['step'], e['loss']) for e in log if 'loss' in e]
kl_entries = [(e['step'], e['kl']) for e in log if 'kl' in e]
eval_reward_entries = [(e['step'], e['eval_reward']) for e in log if 'eval_reward' in e]

print(f'Latest checkpoint: {latest_ckpt}')
print(f'Logged: {len(reward_entries)} reward, {len(loss_entries)} loss, {len(kl_entries)} kl, {len(eval_reward_entries)} eval reward entries')

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

if reward_entries:
    s, r = zip(*reward_entries)
    axes[0].plot(s, r, marker='o', linewidth=2, color='#2E86AB')
    axes[0].set_xlabel('Training Step'); axes[0].set_ylabel('Mean Reward')
    axes[0].set_title('Train Reward')
    axes[0].grid(alpha=0.3)

if loss_entries:
    s, l = zip(*loss_entries)
    axes[1].plot(s, l, marker='o', linewidth=2, color='#A23B72')
    axes[1].axhline(0, linestyle='--', color='gray', alpha=0.5)
    axes[1].set_xlabel('Training Step'); axes[1].set_ylabel('GRPO Loss')
    axes[1].set_title('GRPO Loss')
    axes[1].grid(alpha=0.3)

if kl_entries:
    s, k = zip(*kl_entries)
    axes[2].plot(s, k, marker='o', linewidth=2, color='#F18F01')
    axes[2].set_xlabel('Training Step'); axes[2].set_ylabel('KL Divergence')
    axes[2].set_title('Policy KL from Base')
    axes[2].grid(alpha=0.3)

if eval_reward_entries:
    s, er = zip(*eval_reward_entries)
    axes[3].plot(s, er, marker='o', linewidth=2, color='#4C956C')
    axes[3].set_xlabel('Training Step'); axes[3].set_ylabel('Eval Mean Reward')
    axes[3].set_title('Eval Reward During Training')
    axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/training_curves.png', dpi=140, bbox_inches='tight')
plt.show()
print(f'\nSaved: {RUN_DIR}/training_curves.png')

if reward_entries:
    initial = reward_entries[0][1]
    final = reward_entries[-1][1]
    peak_step, peak = max(reward_entries, key=lambda x: x[1])
    print(f'\nReward: initial={initial:.3f}  final={final:.3f}  peak={peak:.3f} at step {peak_step}')
if kl_entries:
    final_kl = kl_entries[-1][1]
    max_step, max_kl = max(kl_entries, key=lambda x: x[1])
    print(f'KL: final={final_kl:.4f}  max={max_kl:.4f} at step {max_step}')


## Cell 6 — Run held-out evaluation

Evaluates the trained adapter on all 120 held-out v3 traces and compares it to the base model with the same 3-shot prompt. The `easy` and `hard` fields use the same v3 file because the eval script requires both flags; the `hard` field is the reported OOD result.


In [ ]:
import glob
import re

RUN_DIR = '/content/snitch-env/runs/grpo_lr2e5_400steps'

def ckpt_step(path):
    m = re.search(r'checkpoint-(\d+)$', path)
    return int(m.group(1)) if m else -1

ckpts = sorted(glob.glob(f'{RUN_DIR}/checkpoint-*'), key=ckpt_step)
if not ckpts:
    raise FileNotFoundError('No checkpoints found — training cell may have failed')

final_ckpt = ckpts[-1]
print(f'Evaluating LoRA checkpoint: {final_ckpt}')

!mkdir -p /content/snitch-env/results

# LoRA eval on full held-out v3 set
!python scripts/gen_gap_eval.py \
  --model-path {final_ckpt} \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --eval-easy data/eval_traces.jsonl \
  --eval-hard data/eval_traces.jsonl \
  --out /content/snitch-env/results/eval_lora_lr2e5_400.json

# Base model eval with the same 3-shot prompt, same full held-out v3 set
!mkdir -p /tmp/no-adapter
!python scripts/gen_gap_eval.py \
  --model-path /tmp/no-adapter \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --eval-easy data/eval_traces.jsonl \
  --eval-hard data/eval_traces.jsonl \
  --out /content/snitch-env/results/eval_base_full120.json


## Cell 6b — Re-evaluate the original 300-step checkpoint at n=120

This gives a fair three-way comparison on the same full held-out v3 set: base model, original 300-step LoRA, and new 400-step LoRA.


In [ ]:
from huggingface_hub import snapshot_download

old_ckpt = snapshot_download(repo_id='Mihir1107/snitch-overseer-ckpt300')
print(f'Original checkpoint downloaded to: {old_ckpt}')

!python scripts/gen_gap_eval.py \
  --model-path {old_ckpt} \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --eval-easy data/eval_traces.jsonl \
  --eval-hard data/eval_traces.jsonl \
  --out /content/snitch-env/results/eval_old_ckpt_n120.json


## Cell 7 — Display held-out results

Use this table to decide whether the new run should replace the previous checkpoint in the README. It compares base, old LoRA, and new LoRA on the same full n=120 held-out v3 eval.


In [ ]:
import json
from pathlib import Path

with open('/content/snitch-env/results/eval_lora_lr2e5_400.json') as f:
    new_lora = json.load(f)
with open('/content/snitch-env/results/eval_base_full120.json') as f:
    base = json.load(f)

old_path = Path('/content/snitch-env/results/eval_old_ckpt_n120.json')
old_lora = None
if old_path.exists():
    with old_path.open() as f:
        old_lora = json.load(f)

# 'hard' field is the actual held-out v3 result; 'easy' is the same data passed twice for script compatibility.
def row_metrics(result):
    return (
        result['hard']['accuracy'],
        result['hard']['mean_reward'],
        result['hard']['parse_error_rate'],
    )

base_acc, base_rew, base_pe = row_metrics(base)
new_acc, new_rew, new_pe = row_metrics(new_lora)
if old_lora:
    old_acc, old_rew, old_pe = row_metrics(old_lora)

print('=' * 80)
print('GENERALIZATION ON HELD-OUT v3 (30 unseen questions × 4 classes = 120 traces)')
print('=' * 80)
print(f'\n                       Accuracy   Mean Reward   Parse Errors')
print(f'Random baseline:       25.0%      —             —')
print(f'Base + 3-shot:         {base_acc*100:5.1f}%      {base_rew:+.3f}        {base_pe*100:.1f}%')
if old_lora:
    print(f'Old LoRA ckpt300:      {old_acc*100:5.1f}%      {old_rew:+.3f}        {old_pe*100:.1f}%')
print(f'New LoRA lr2e5/400:    {new_acc*100:5.1f}%      {new_rew:+.3f}        {new_pe*100:.1f}%')
print()
print('Lift vs base:')
print(f'  New accuracy:        {(new_acc-base_acc)*100:+.1f}pp')
print(f'  New mean reward:     {new_rew-base_rew:+.3f}')
if old_lora:
    print()
    print('New vs old checkpoint:')
    print(f'  Accuracy:            {(new_acc-old_acc)*100:+.1f}pp')
    print(f'  Mean reward:         {new_rew-old_rew:+.3f}')
print('=' * 80)

print('\nRaw new LoRA JSON:')
print(json.dumps(new_lora, indent=2))
print('\nRaw base JSON:')
print(json.dumps(base, indent=2))
if old_lora:
    print('\nRaw old LoRA JSON:')
    print(json.dumps(old_lora, indent=2))


## Cell 8 — Upload trained adapter to Hugging Face Hub

Run this only if the held-out metrics above are worth publishing. Add an `HF_TOKEN` Colab secret first.


In [ ]:
import os
import glob
import re
import json
from google.colab import userdata
from huggingface_hub import HfApi

RUN_DIR = '/content/snitch-env/runs/grpo_lr2e5_400steps'
REPO_ID = 'Mihir1107/snitch-overseer-lr2e5-ckpt400'
QUALITY_MIN_ACCURACY = 0.60
QUALITY_MAX_PARSE_ERROR = 0.01

with open('/content/snitch-env/results/eval_lora_lr2e5_400.json') as f:
    new_results = json.load(f)

new_acc = new_results['hard']['accuracy']
new_parse = new_results['hard']['parse_error_rate']

if not (new_acc >= QUALITY_MIN_ACCURACY and new_parse <= QUALITY_MAX_PARSE_ERROR):
    print(f'NOT uploading. Accuracy={new_acc:.3f}, parse_errors={new_parse:.3f}')
    print('New run did not exceed quality threshold. Keep existing submission.')
else:
    def ckpt_step(path):
        m = re.search(r'checkpoint-(\d+)$', path)
        return int(m.group(1)) if m else -1

    ckpts = sorted(glob.glob(f'{RUN_DIR}/checkpoint-*'), key=ckpt_step)
    if not ckpts:
        raise FileNotFoundError('No checkpoints found — training cell may have failed')

    final_ckpt = ckpts[-1]
    print(f'Uploading adapter from: {final_ckpt}')
    print(f'Target repo: https://huggingface.co/{REPO_ID}')

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    if not os.environ.get('HF_TOKEN'):
        raise RuntimeError('HF_TOKEN Colab secret is missing')

    api = HfApi()
    api.create_repo(repo_id=REPO_ID, repo_type='model', private=False, exist_ok=True)
    api.upload_folder(
        folder_path=final_ckpt,
        repo_id=REPO_ID,
        repo_type='model',
    )
    print(f'Uploaded — meets quality bar. Repo: https://huggingface.co/{REPO_ID}')


## Done

Artifacts produced:

- `runs/grpo_lr2e5_400steps/checkpoint-{50,100,150,200,250,300,350,400}/` — LoRA adapters at each savepoint
- `runs/grpo_lr2e5_400steps/training_curves.png` — reward, loss, KL, and eval-reward plots
- `results/eval_lora_lr2e5_400.json` — new LoRA on all held-out v3 traces
- `results/eval_base_full120.json` — base-model A/B control on all held-out v3 traces
- `results/eval_old_ckpt_n120.json` — original 300-step LoRA re-evaluated on all held-out v3 traces
- Optional HF repo if quality bar passes: `Mihir1107/snitch-overseer-lr2e5-ckpt400`

If this run beats the previous checkpoint, update README/blog/results/figure with the new numbers and link to the new adapter repo.
